# Part 5
## 1. 누락 데이터 처리

### 1-1 누락 데이터 확인

- `sns.load_dataset(’데이터셋 이름’)`: seaborn에 내장된 데이터셋을 DataFrame으로 불러오기
- `Series.value_counts(dropna=True 또는 False)` : 시리즈 안의 고유한 값별로 횟수를 계산해 값을 인덱스로 하고 휫수를 값으로 하는 시리즈를 반환
    - 횟수가 큰 순서로 정렬
    - `dropna=True`: Nan 세지 않음
    - `dropna=False`: Nan 포함
- `isnull()` :
    - Nan →True 반환, 1로 판별
    - 데이터 존재 →False 반환, 0으로 판별
- `notnull()` : 데이터 존재 →False 반환, Nan →True 반환

In [23]:
import seaborn as sns

df = sns.load_dataset('titanic')
df.head()
df.info()

#deck 열의 Nan 개수 계산하기
nan_deck = df['deck'].value_counts(dropna=False)
print(nan_deck)

#insnull() 메소드로 누락 데이터 찾기
print(df.head().isnull())

#notnull() 메소드로 누락 데이터 찾기
print(df.head().notnull())

#isnull() 메소드로 누락 데이터 개수 구하기
print(df.head().isnull().sum(axis=0))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
deck
NaN    688
C       59
B       47
D       33
E       32
A       15
F       

### 1-2 누락 데이터 제거
- `DataFrame.dropna(axis=0 또는 1, how='any' 또는 'all', thresh=정수, subset=[열이름1, 열이름2], inplace=False)` : NaN이 포함된 행 또는 열을 삭제하고, 삭제된 결과를 새 DataFrame으로 반환
    
    
    | 매개변수 | 정의 |
    | --- | --- |
    | `axis` | `0`이면 행 단위, `1`이면 열 단위로 NaN을 검사해 삭제 (기본값 0) |
    | `how` | `'any'`: 검사 대상 중 NaN이 **하나라도** 있으면 삭제 (기본값). `'all'`: **전부** NaN일 때만 삭제 |
    | `thresh` | 한 행(또는 열)에서 **NaN이 아닌 값이 최소 정수 개 이상**이어야 남김. 그보다 적으면 삭제 |
    | `subset` | NaN 검사 대상을 지정한 열들로 한정 (`axis=0`일 때 사용) |
    | `inplace` | `True`면 원본을 직접 수정, `False`면 새 객체 반환 (기본값) |

In [24]:
import seaborn as sns

df = sns.load_dataset('titanic')

#for 반복문으로 각 열의 NaN 개수 계산
missing_df = df.isnull()
for col in missing_df.columns:
    missing_count = missing_df[col].value_counts()
    #try 블록을 실행하다 오류가 발생하면 프로그램을 멈추지 않고 except 블록을 실행
    try:
        print(col, ': ', missing_count[True]) #NaN 값이 있으면 개수 출력
    except:
        print(col, ': ', 0) #NaN 값이 없으면 0개 출력

#열 단위로 검사해 NaN이 아닌 값이 500개 미만인 열을 삭제 ->deck 삭제됨
df_thresh = df.dropna(axis=1, thresh=500)
print(df_thresh.columns)

#age 열에 나이 데이터가 없는 모든 행 삭제
df_age = df.dropna(subset=['age'], how='any', axis=0)
print(len(df_age))

survived :  0
pclass :  0
sex :  0
age :  177
sibsp :  0
parch :  0
fare :  0
embarked :  2
class :  0
who :  0
adult_male :  0
deck :  688
embark_town :  2
alive :  0
alone :  0
Index(['survived', 'pclass', 'sex', 'age', 'sibsp', 'parch', 'fare',
       'embarked', 'class', 'who', 'adult_male', 'embark_town', 'alive',
       'alone'],
      dtype='object')
714


### 1-3 누락 데이터 치환

- `Series.fillna(채울값, inplace=False)`: NaN을 지정한 값으로 변경
    - `method=’ffill’` : 바로 앞 행 값으로 채움
    - `method=’bfill’` : 뒤 행의 값으로 채움


In [25]:
import seaborn as sns

df = sns.load_dataset('titanic')

#age 열의 첫 10개 데이터 출력
print(df['age'].head(10))
print('\n')

#age 열의 NaN 값을 다른 나이 데이터의 평균으로 변경
mean_age = df['age'].mean(axis=0)
df['age'].fillna(mean_age, inplace=True)

#변경된 값이 반영된 age열의 첫 10개 데이터 출력
print(df['age'].head(10))

0    22.0
1    38.0
2    26.0
3    35.0
4    35.0
5     NaN
6    54.0
7     2.0
8    27.0
9    14.0
Name: age, dtype: float64


0    22.000000
1    38.000000
2    26.000000
3    35.000000
4    35.000000
5    29.699118
6    54.000000
7     2.000000
8    27.000000
9    14.000000
Name: age, dtype: float64


/tmp/ipykernel_1257/4174308429.py:11: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['age'].fillna(mean_age, inplace=True)


In [26]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import seaborn as sns

# titanic 데이터셋 가져오기
df = sns.load_dataset('titanic')

# embark_town 열의 829행의 NaN 데이터 출력
print(df['embark_town'][825:830])
print('\n')

# embark_town 열의 NaN값을 승선도시 중에서 가장 많이 출현한 값으로 치환하기
most_freq = df['embark_town'].value_counts(dropna=True).idxmax()
print(most_freq)
print('\n')

df['embark_town'] = df['embark_town'].fillna(most_freq)

# embark_town 열 829행의 NaN 데이터 출력(NaN값이 most_freq 값으로 대체)
print(df['embark_town'][825:830])

825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829            NaN
Name: embark_town, dtype: object


Southampton


825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829    Southampton
Name: embark_town, dtype: object


- `Series객체.fillna(method='ffill' 또는 'bfill', inplace=True 또는 False)`
    - `method=’ffill’` : 바로 앞 행 값으로 채움
    - `method=’bfill’` : 뒤 행의 값으로 채움
    - `inplace=True` : 원본객체 변경하기

In [27]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import seaborn as sns

# titanic 데이터셋 가져오기
df = sns.load_dataset('titanic')

# embark_town 열의 829행의 NaN 데이터 출력
print(df['embark_town'][825:830])
print('\n')

# embark_town 열의 NaN값을 바로 앞에 있는 828행의 값으로 변경하기
df['embark_town'].fillna(method='ffill', inplace=True)
print(df['embark_town'][825:830])

825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829            NaN
Name: embark_town, dtype: object


825     Queenstown
826    Southampton
827      Cherbourg
828     Queenstown
829     Queenstown
Name: embark_town, dtype: object


/tmp/ipykernel_1257/1571798342.py:14: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['embark_town'].fillna(method='ffill', inplace=True)
/tmp/ipykernel_1257/1571798342.py:14: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['embark_town'].fillna(method='ffill', inplace=True)


## 2. 중복 데이터 처리

### 2-1 중복 데이터 확인

- `duplicated()` : 각 행의 중복 여부를 나타내는 불리언 시리즈 반환
    - 중복되는 행: `True` 반환
    - 처음 나오는 행: `False` 반환
    - 데이터프레임에 적용: 모든 열의 값이 같을 때 중복으로 판단
- `저장할변수 = DataFrame객체['열이름'].duplicated()` : 지정한 열의 각 원소에 대해 중복 여부를 판별해 원래 열과 같은 길이의 불리언 시리즈를 저장

In [28]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# 중복 데이터를 갖는 데이터프레임 만들기
df = pd.DataFrame({'c1':['a', 'a', 'b', 'a', 'b'],
                   'c2':[1, 1, 1, 2, 2],
                   'c3':[1, 1, 2, 2, 2]})
print(df)
print('\n')

# 데이터프레임 전체 행 데이터 중에서 중복값 찾기
df_dup = df.duplicated()
print(df_dup)
print('\n')

# 데이터프레임의 특정 열 데이터에서 중복값 찾기
col_dup = df['c2'].duplicated()
print(col_dup)

  c1  c2  c3
0  a   1   1
1  a   1   1
2  b   1   2
3  a   2   2
4  b   2   2


0    False
1     True
2    False
3    False
4    False
dtype: bool


0    False
1     True
2     True
3    False
4     True
Name: c2, dtype: bool


### 2-2 중복 데이터 제거

- `drop_duplicates()` : 중복되는 행 제거, 고유한 관측값을 가진 행만 남긴 데이터프레임을 반환
    - `DataFrame객체.drop_duplicates(subset=[열이름1, 열이름2, ...], inplace=True 또는 False)`
        
        
        | 옵션 | 의미 |
        | --- | --- |
        | (옵션 없음) | 모든 열의 값이 같은 행을 중복으로 판단 |
        | `subset=[열이름 리스트]` | 리스트에 지정한 열들의 값만 비교해서 중복 여부를 판단 |
        | `inplace=True` | 원본 객체를 직접 변경 (기본값 `False`는 새로운 객체를 반환) |
    - 중복되는 행들 중 처음 나오는 행은 남김, 그 뒤에 나오는 중복 행 제거


In [29]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# 중복 데이터를 갖는 데이터프레임 만들기
df = pd.DataFrame({'c1':['a', 'a', 'b', 'a', 'b'],
                   'c2':[1, 1, 1, 2, 2],
                   'c3':[1, 1, 2, 2, 2]})
print(df)
print('\n')

# 데이터프레임에서 중복 행 제거
df2 = df.drop_duplicates()
print(df2)
print('\n')

# c2, c3열을 기준으로 중복 행 제거
df3 = df.drop_duplicates(subset=['c2', 'c3'])
print(df3)

  c1  c2  c3
0  a   1   1
1  a   1   1
2  b   1   2
3  a   2   2
4  b   2   2


  c1  c2  c3
0  a   1   1
2  b   1   2
3  a   2   2
4  b   2   2


  c1  c2  c3
0  a   1   1
2  b   1   2
3  a   2   2


## 3. 데이터 표준화

### 3-1 단위 환산

- `round()`: 지정한 소수점 아래 자리수까지 남기고, 그 다음 자리에서 반올림
    - ex. `round(2)`: 소수점 둘째자리에서 반올림
    
    > `Series객체.round(소수점아래자릿수)`
    >

In [6]:
# -*- coding: utf-8 -*-
# 라이브러리 불러오기
import pandas as pd
import matplotlib.pyplot as plt

# 구글 드라이브 마운트 (최초 1회 인증 진행)
from google.colab import drive
drive.mount('/content/drive')

# CSV 데이터를 데이터프레임으로 변환
# 주의: 파일이 '내 드라이브' 루트가 아닌 특정 폴더 안에 있다면 폴더 경로를 포함해야 합니다.
# 예: '/content/drive/MyDrive/데이터/auto-mpg.csv'
file_path = '/content/drive/MyDrive/학회/part 5 data/auto-mpg.csv'

df = pd.read_csv(file_path, header=0)
df.head()

# 열 이름 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']
print(df.head(3))
print('\n')

# mpg(mile per gallon)를 kpl(kilometer per liter)로 변환(mpg_to_kpl = 0.425)
mpg_to_kpl = 1.60934/3.78541

# mpg 열에 0.425를 곱한 결과를 새로운 열(kpl)에 추가
df['kpl'] = df['mpg'] * mpg_to_kpl
print(df.head(3))
print('\n')

# kpl 열을 소수점 아래 둘째자리에서 반올림
df['kpl'] = df['kpl'].round(2)
print(df.head(3))

Mounted at /content/drive
    mpg  cylinders  displacement horsepower  weight  acceleration  model year  \
0  15.0          8         350.0      165.0  3693.0          11.5          70   
1  18.0          8         318.0      150.0  3436.0          11.0          70   
2  16.0          8         304.0      150.0  3433.0          12.0          70   

   origin                name  
0       1   buick skylark 320  
1       1  plymouth satellite  
2       1       amc rebel sst  


    mpg  cylinders  displacement horsepower  weight  acceleration  model year  \
0  15.0          8         350.0      165.0  3693.0          11.5          70   
1  18.0          8         318.0      150.0  3436.0          11.0          70   
2  16.0          8         304.0      150.0  3433.0          12.0          70   

   origin                name       kpl  
0       1   buick skylark 320  6.377143  
1       1  plymouth satellite  7.652571  
2       1       amc rebel sst  6.802286  


    mpg  cylinders  disp

### 3-2 자료형 변환

- `unique()` : 시리즈에서 중복을 제거한 고유값을 처음 나타난 순서의 배열로 반환
    
    > `Series객체.unique()`
    >
- `replace()` :
    - `Series객체.replace(바꿀값, 새로운값, inplace=True 또는 False)`
        
        →값 하나를 다른 값으로 변경
        
    - `Series객체.replace({바꿀값1:새로운값1, 바꿀값2:새로운값2, ...}, inplace=True 또는 False)`
        
        →딕셔너리 키:값 쌍으로 여러 값을 한 번에 바꿈
        
- `astype()` : 시리즈의 자료형을 지정한 자료형으로 변환
    
    > `Series객체.astype('자료형')`
    - 자료형 문자열: ‘float’(실수형), ‘int’(정수형), ‘category’(범주형), ‘str’(문자형)
- `Series객체.sample(n)` : 시리즈에서 n개의 원소를 무작위로 추출해선 반환

In [7]:
# -*- coding: utf-8 -*-
# 라이브러리 불러오기
import pandas as pd
import matplotlib.pyplot as plt

# 구글 드라이브 마운트 (최초 1회 인증 진행)
from google.colab import drive
drive.mount('/content/drive')

# CSV 데이터를 데이터프레임으로 변환
# 주의: 파일이 '내 드라이브' 루트가 아닌 특정 폴더 안에 있다면 폴더 경로를 포함해야 합니다.
# 예: '/content/drive/MyDrive/데이터/auto-mpg.csv'
file_path = '/content/drive/MyDrive/학회/part 5 data/auto-mpg.csv'

df = pd.read_csv(file_path, header=0)
df.head()

# 열 이름 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']

# 각 열의 자료형 확인
print(df.dtypes)
print('\n')

# horsepower 열의 고유값 확인
print(df['horsepower'].unique())

# 누락 데이터('?') 삭제
import numpy as np
df['horsepower'].replace('?', np.nan, inplace=True)   # '?'을 np.nan으로 변경
df.dropna(subset=['horsepower'], axis=0, inplace=True)   # 누락 데이터 행 삭제
df['horsepower'] = df['horsepower'].astype('float')   # 문자열을 실수형으로 변환

# horsepower 열의 자료형 확인
print(df['horsepower'].dtypes)
print('\n')

# origin 열의 고유값 확인
print(df['origin'].unique())

# 정수형 데이터를 문자형 데이터로 변환
df['origin'] = df['origin'].replace({1:'USA', 2:'EU', 3:'JPN'},inplace=True)

# 정수형 데이터를 문자형 데이터로 변환
df['origin'].replace({1:'USA', 2:'EU', 3:'JPN'}, inplace=True)

# origin 열의 고유값과 자료형 확인
print(df['origin'].unique())
print(df['origin'].dtypes)

# 문자열을 범주형으로 변환
df['origin'] = df['origin'].astype('category')
print(df['origin'].dtypes)

# 범주형을 문자열로 다시 변환
df['origin'] = df['origin'].astype('str')
print(df['origin'].dtypes)
print('\n')

# model year 열의 정수형을 범주형으로 변환
print(df['model year'].sample(3))
df['model year'] = df['model year'].astype('category')
print(df['model year'].sample(3))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
mpg             float64
cylinders         int64
displacement    float64
horsepower       object
weight          float64
acceleration    float64
model year        int64
origin            int64
name             object
dtype: object


['165.0' '150.0' '140.0' '198.0' '220.0' '215.0' '225.0' '190.0' '170.0'
 '160.0' '95.00' '97.00' '85.00' '88.00' '46.00' '87.00' '90.00' '113.0'
 '200.0' '210.0' '193.0' '?' '100.0' '105.0' '175.0' '153.0' '180.0'
 '110.0' '72.00' '86.00' '70.00' '76.00' '65.00' '69.00' '60.00' '80.00'
 '54.00' '208.0' '155.0' '130.0' '112.0' '92.00' '145.0' '137.0' '158.0'
 '167.0' '94.00' '107.0' '230.0' '49.00' '75.00' '91.00' '122.0' '67.00'
 '83.00' '78.00' '52.00' '61.00' '93.00' '148.0' '129.0' '96.00' '71.00'
 '98.00' '115.0' '53.00' '81.00' '79.00' '120.0' '152.0' '102.0' '108.0'
 '68.00' '58.00' '149.0' '89.00' '63.00' '48.00' '66.00' '1

/tmp/ipykernel_1257/642816327.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['horsepower'].replace('?', np.nan, inplace=True)   # '?'을 np.nan으로 변경
/tmp/ipykernel_1257/642816327.py:43: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(val

## 4. 범주형 데이터 처리

### 4-1 구간 분할

- 구간 분할(binning): 연속 변수를 일정한 구간으로 나누고 각 구간을 범주형 이산 변수로 변환하는 과정
- np.histogram()
    
    > `개수변수, 경계값변수 = np.histogram(데이터배열, bins=구간개수)`
    >
    
    →데이터를 bins에 지정한 개수의 구간으로 나누고 두 값을 반환
    
    - 첫 번째 값: 각 구간에 속하는 값의 개수
    - 두 번째 값: 구간의 경계값 리스트. 구간이 n개이면 경계값은 n+1개
  - `pd.cut()`
    
    > `pd.cut(x=데이터배열, bins=경계값리스트, labels=구간이름리스트, include_lowest=True 또는 False)`
    >
    
    →연속 데이터를 경계값 리스트 기준으로 구간에 할당하고 각 값에 해당 구간의 이름을 붙여 범주형 데이터로 반환
    
    | 옵션 | 의미 |
    | --- | --- |
    | `x` | 구간으로 나눌 데이터 배열 |
    | `bins` | 구간의 경계값 리스트 |
    | `labels` | 각 구간의 이름 리스트 (구간 개수와 같아야 함) |
    | `include_lowest=True` | 첫 구간의 낮은 경계값(최솟값)을 포함 |

In [8]:
# -*- coding: utf-8 -*-
# 라이브러리 불러오기
import pandas as pd
import matplotlib.pyplot as plt

# 구글 드라이브 마운트 (최초 1회 인증 진행)
from google.colab import drive
drive.mount('/content/drive')

# CSV 데이터를 데이터프레임으로 변환
# 주의: 파일이 '내 드라이브' 루트가 아닌 특정 폴더 안에 있다면 폴더 경로를 포함해야 합니다.
# 예: '/content/drive/MyDrive/데이터/auto-mpg.csv'
file_path = '/content/drive/MyDrive/학회/part 5 data/auto-mpg.csv'

# 열 이름 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']

# horsepower 열의 누락 데이터('?') 삭제하고 실수형으로 변환
df['horsepower'].replace('?', np.nan, inplace=True)   # '?'을 np.nan으로 변경
df.dropna(subset=['horsepower'], axis=0, inplace=True)   # 누락 데이터 행 삭제
df['horsepower'] = df['horsepower'].astype('float')   # 문자열을 실수형으로 변환

# np.histogram 함수로 3개의 bin으로 나누는 경계값의 리스트 구하기
count, bin_dividers = np.histogram(df['horsepower'], bins=3)
print(bin_dividers)

# 3개의 bin에 이름 지정
bin_names = ['저출력', '보통출력', '고출력']

# pd.cut으로 각 데이터를 3개의 bin에 할당
df['hp_bin'] = pd.cut(x=df['horsepower'],     # 데이터 배열
                      bins=bin_dividers,      # 경계값 리스트
                      labels=bin_names,       # bin 이름
                      include_lowest=True)    # 첫 경계값 포함

# horsepower 열, hp_bin 열의 첫 15행 출력
print(df[['horsepower', 'hp_bin']].head(15))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[ 46.         107.33333333 168.66666667 230.        ]
    horsepower hp_bin
0        165.0   보통출력
1        150.0   보통출력
2        150.0   보통출력
3        140.0   보통출력
4        198.0    고출력
5        220.0    고출력
6        215.0    고출력
7        225.0    고출력
8        190.0    고출력
9        170.0    고출력
10       160.0   보통출력
11       150.0   보통출력
12       225.0    고출력
13        95.0    저출력
14        95.0    저출력


/tmp/ipykernel_1257/2828729830.py:20: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['horsepower'].replace('?', np.nan, inplace=True)   # '?'을 np.nan으로 변경


### 4-2 더미 변수

- 더미 변수: 0 또는 1로 표현되는 변수
    - 0: 해당 특성이 존재하지 않음
    - 1: 해당 특성이 존재함
- 원핫인코딩: 범주형 데이터를 0과 1로만 구성된 원핫벡터로 변환하는 것
- `pd.get_dummies(범주형시리즈, dtype=자료형)` : 범주형 시리즈의 고유값을 새로운 열로 만든 데이터프레임을 반환

In [9]:
# 3개의 bin에 이름 지정
bin_names = ['저출력', '보통출력', '고출력']

# pd.cut으로 각 데이터를 3개의 bin에 할당
df['hp_bin'] = pd.cut(x=df['horsepower'],     # 데이터 배열
                      bins=bin_dividers,      # 경계값 리스트
                      labels=bin_names,       # bin 이름
                      include_lowest=True)    # 첫 경계값 포함

# hp_bin 열의 범주형 데이터를 더미 변수로 변환
horsepower_dummies = pd.get_dummies(df['hp_bin'],dtype=int)
print(horsepower_dummies.head(15))

    저출력  보통출력  고출력
0     0     1    0
1     0     1    0
2     0     1    0
3     0     1    0
4     0     0    1
5     0     0    1
6     0     0    1
7     0     0    1
8     0     0    1
9     0     0    1
10    0     1    0
11    0     1    0
12    0     0    1
13    1     0    0
14    1     0    0


- `from 라이브러리 import 모듈` : 라이브러리에서 지정한 모듈만 불러옴
    - sklearn 라이브러리: 원핫인코딩을 편하게 처리 가능
        - 범주형 데이터를 원핫벡터로 변환
        - 결과는 희소행렬로 정리
- `preprocessing.LabelEncoder()` : 문자열 범주를 0부터 시작하는 정수로 바꾸는 객체를 만듦
- `preprocessing.OneHotEncoder` : 정수 범주를 원핫벡터로 바꾸는 인코더 객체를 만듦
    - 변환결과: 희소행렬…
- `인코더객체.fit_transform(데이터)` : 데이터에 맞춰 변환 규칠을 학습하고 그 규칙으로 변환한 결과를 반환
- `넘파이배열.reshape(행개수, 열개수)` : 배열의 원소 개수는 그대로 두고, 지정한 행·열 개수의 형태로 바꾼 배열을 반환

In [10]:
# sklern 라이브러리 불러오기
from sklearn import preprocessing

# 전처리를 위한 encoder 객체 만들기
label_encoder = preprocessing.LabelEncoder()        # label encoder 생성
onehot_encoder = preprocessing.OneHotEncoder()      # one hot encoder 생성

# label encoder로 문자열 범주를 숫자형 범주로 변환
onehot_labeled = label_encoder.fit_transform(df['hp_bin'].head(15))
print(onehot_labeled)
print(type(onehot_labeled))

# 2차원 행렬 형태로 변경
onehot_reshaped = onehot_labeled.reshape(len(onehot_labeled), 1)
print(onehot_reshaped)
print(type(onehot_reshaped))

# 희소행렬로 변환
onehot_fitted = onehot_encoder.fit_transform(onehot_reshaped)
print(onehot_fitted)
print(type(onehot_fitted))

[1 1 1 1 0 0 0 0 0 0 1 1 0 2 2]
<class 'numpy.ndarray'>
[[1]
 [1]
 [1]
 [1]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [1]
 [1]
 [0]
 [2]
 [2]]
<class 'numpy.ndarray'>
<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 15 stored elements and shape (15, 3)>
  Coords	Values
  (0, 1)	1.0
  (1, 1)	1.0
  (2, 1)	1.0
  (3, 1)	1.0
  (4, 0)	1.0
  (5, 0)	1.0
  (6, 0)	1.0
  (7, 0)	1.0
  (8, 0)	1.0
  (9, 0)	1.0
  (10, 1)	1.0
  (11, 1)	1.0
  (12, 0)	1.0
  (13, 2)	1.0
  (14, 2)	1.0
<class 'scipy.sparse._csr.csr_matrix'>


## 5. 정규화

> 정규화란? 각 려에 속하는 데이터 값을 동일한 크기 기준으로 나눈 비율로 나타내는 것
>
- 정규화를 거친 데이터의 범위: 0 ~ 1 or -1 ~ 1
- `describe()` : 데이터의 통계 요약 정보 반환, 시리즈에 적용하면 다음 항목이 시리즈로 나옴↓↓↓↓↓↓↓
    
    
    | 항목 | 의미 |
    | --- | --- |
    | `count` | 데이터 개수 (NaN 제외) |
    | `mean` | 평균 |
    | `std` | 표준편차 |
    | `min` | 최솟값 |
    | `25%` | 하위 25% 지점의 값 (제1사분위수) |
    | `50%` | 하위 50% 지점의 값 (중앙값) |
    | `75%` | 하위 75% 지점의 값 (제3사분위수) |
    | `max` | 최댓값 |
- abs() : 숫자의 절대값을 반환

In [11]:
import pandas as pd
import numpy as np

# read_csv() 함수로 df 생성
df = pd.read_csv('/content/drive/MyDrive/학회/part 5 data/auto-mpg.csv', header=None)

# 열 이름 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']

# horsepower 열의 누락 데이터('?') 삭제하고 실수형으로 변환
df['horsepower'] = df['horsepower'].replace('?', np.nan)   # '?'을 np.nan으로 변경
df.dropna(subset=['horsepower'], axis=0, inplace=True)   # 누락 데이터 행 삭제
df['horsepower'] = df['horsepower'].astype('float')   # 문자열을 실수형으로 변환

# horsepower 열의 통계 요약정보로 최대값(max) 확인
print(df.horsepower.describe())
print('\n')

# horsepower 열의 최대값의 절대값으로 모든 데이터를 나누어 저장
df.horsepower = df.horsepower/abs(df.horsepower.max())

print(df.horsepower.head())
print('\n')
print(df.horsepower.describe())

count    392.000000
mean     104.469388
std       38.491160
min       46.000000
25%       75.000000
50%       93.500000
75%      126.000000
max      230.000000
Name: horsepower, dtype: float64


0    0.565217
1    0.717391
2    0.652174
3    0.652174
4    0.608696
Name: horsepower, dtype: float64


count    392.000000
mean       0.454215
std        0.167353
min        0.200000
25%        0.326087
50%        0.406522
75%        0.547826
max        1.000000
Name: horsepower, dtype: float64


In [12]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd
import numpy as np

# read_csv() 함수로 df 생성
df = pd.read_csv('/content/drive/MyDrive/학회/part 5 data/auto-mpg.csv', header=None)

# 열 이름 지정
df.columns = ['mpg','cylinders','displacement','horsepower','weight',
              'acceleration','model year','origin','name']

# horsepower 열의 누락 데이터('?') 삭제하고 실수형으로 변환
df['horsepower'].replace('?', np.nan, inplace=True)   # '?'을 np.nan으로 변경
df.dropna(subset=['horsepower'], axis=0, inplace=True)   # 누락 데이터 행 삭제
df['horsepower'] = df['horsepower'].astype('float')   # 문자열을 실수형으로 변환

# horsepower 열의 통계 요약정보로 최대값(max)과 최소값(min) 확인
print(df.horsepower.describe())
print('\n')

# horsepower 열의 최대값의 절대값으로 모든 데이터를 나누어 저장
min_x = df.horsepower - df.horsepower.min() #모든 값에서 최솟값 뺀 값을 분자에 저장
min_max = df.horsepower.max() - df.horsepower.min() #분모 계산
df.horsepower = min_x/min_max

print(df.horsepower.head())
print('\n')
print(df.horsepower.describe())

count    392.000000
mean     104.469388
std       38.491160
min       46.000000
25%       75.000000
50%       93.500000
75%      126.000000
max      230.000000
Name: horsepower, dtype: float64


0    0.456522
1    0.646739
2    0.565217
3    0.565217
4    0.510870
Name: horsepower, dtype: float64


count    392.000000
mean       0.317768
std        0.209191
min        0.000000
25%        0.157609
50%        0.258152
75%        0.434783
max        1.000000
Name: horsepower, dtype: float64


/tmp/ipykernel_1257/515446220.py:15: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['horsepower'].replace('?', np.nan, inplace=True)   # '?'을 np.nan으로 변경


## 6. 시계열 데이터

> Timestamp: 특정한 시점을 기록
>
>
> Period: 두 시점 사이의 일정한 기간
>

### 6-1 다른 자료형을 시계열 객체로 변환
####문자열을 Timestamp로 변환

- `pd.to_datetime()` : 문자열 등 다른 자료형을 판다스 TImestamp를 나타내는 datetime64 자료형으로 변환
- `DataFrame객체.set_index('열이름', inplace=True 또는 False)` : 지정한 열을 행 인덱스로 만듦
    - Timestamp로 된 열을 인덱스로 지정하면 →DatatimeIndex가 됨

In [13]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# read_csv() 함수로 CSV 파일을 가져와서 df로 변환
df = pd.read_csv('/content/drive/MyDrive/학회/part 5 data/stock-data.csv')

# 데이터 내용 및 자료형 확인
print(df.head())
print('\n')
print(df.info())

         Date  Close  Start   High    Low  Volume
0  2018-07-02  10100  10850  10900  10000  137977
1  2018-06-29  10700  10550  10900   9990  170253
2  2018-06-28  10400  10900  10950  10150  155769
3  2018-06-27  10900  10800  11050  10500  133548
4  2018-06-26  10800  10900  11000  10700   63039


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Date    20 non-null     object
 1   Close   20 non-null     int64 
 2   Start   20 non-null     int64 
 3   High    20 non-null     int64 
 4   Low     20 non-null     int64 
 5   Volume  20 non-null     int64 
dtypes: int64(5), object(1)
memory usage: 1.1+ KB
None


In [14]:
# 문자열 데이터(시리즈 객체)를 판다스 Timestamp로 변환
df['new_Date'] = pd.to_datetime(df['Date'])   # df에 새로운 열로 추가

# 데이터 내용 및 자료형 확인
print(df.head())
print('\n')
print(df.info())
print('\n')
print(type(df['new_Date'][0]))

         Date  Close  Start   High    Low  Volume   new_Date
0  2018-07-02  10100  10850  10900  10000  137977 2018-07-02
1  2018-06-29  10700  10550  10900   9990  170253 2018-06-29
2  2018-06-28  10400  10900  10950  10150  155769 2018-06-28
3  2018-06-27  10900  10800  11050  10500  133548 2018-06-27
4  2018-06-26  10800  10900  11000  10700   63039 2018-06-26


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   Date      20 non-null     object        
 1   Close     20 non-null     int64         
 2   Start     20 non-null     int64         
 3   High      20 non-null     int64         
 4   Low       20 non-null     int64         
 5   Volume    20 non-null     int64         
 6   new_Date  20 non-null     datetime64[ns]
dtypes: datetime64[ns](1), int64(5), object(1)
memory usage: 1.2+ KB
None


<class 'pandas._libs.tslibs.timestamps.T

In [15]:
# 시계열 값으로 변환된 열을 새로운 행 인덱스로 지정. 기존 날짜 열은 삭제
df.set_index('new_Date', inplace=True)
df.drop('Date', axis=1, inplace=True)

# 데이터 내용 및 자료형 확인
print(df.head())
print('\n')
print(df.info())

            Close  Start   High    Low  Volume
new_Date                                      
2018-07-02  10100  10850  10900  10000  137977
2018-06-29  10700  10550  10900   9990  170253
2018-06-28  10400  10900  10950  10150  155769
2018-06-27  10900  10800  11050  10500  133548
2018-06-26  10800  10900  11000  10700   63039


<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 20 entries, 2018-07-02 to 2018-06-01
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   Close   20 non-null     int64
 1   Start   20 non-null     int64
 2   High    20 non-null     int64
 3   Low     20 non-null     int64
 4   Volume  20 non-null     int64
dtypes: int64(5)
memory usage: 960.0 bytes
None


#### Timestamp를 Period로 변환

- `to_period()` : Timestamp 객체를 Period 객체로 변환
    - `DatetimeIndex객체.to_period(freq='기간기준')`
        - `freq` 옵션: 기준이 되는 기간 설정
| 옵션 | 설명 | 옵션 | 설명 |
| --- | --- | --- | --- |
| `D` | day(1일) | `B` | business day(휴일 제외) |
| `W` | week(1주) | `H` | hour(1시간) |
| `M` | month end(월말) | `T` | minute(1분) |
| `MS` | month begin(월초) | `S` | second(1초) |
| `Q` | quarter end(분기말) | `L` | millisecond(1/1,000초) |
| `QS` | quarter begin(분기초) | `U` | microsecond(1/1,000,000초) |
| `A` | year end(연말) | `N` | nanosecond(1/1,000,000,000초) |
| `AS` | year begin(연초) |  |  |

In [16]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# 날짜 형식의 문자열로 구성되는 리스트 정의
dates = ['2019-01-01', '2020-03-01', '2021-06-01']

# 문자열의 배열(시리즈 객체)을 판다스 Timestamp로 변환
ts_dates = pd.to_datetime(dates)
print(ts_dates)
print('\n')

# Timestamp를 Period로 변환
pr_day = ts_dates.to_period(freq='D')
print(pr_day)
pr_month = ts_dates.to_period(freq='M')
print(pr_month)
pr_year = ts_dates.to_period(freq='A')
print(pr_year)

DatetimeIndex(['2019-01-01', '2020-03-01', '2021-06-01'], dtype='datetime64[ns]', freq=None)


PeriodIndex(['2019-01-01', '2020-03-01', '2021-06-01'], dtype='period[D]')
PeriodIndex(['2019-01', '2020-03', '2021-06'], dtype='period[M]')
PeriodIndex(['2019', '2020', '2021'], dtype='period[Y-DEC]')


/tmp/ipykernel_1257/1998758328.py:19: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'Y' instead.
  pr_year = ts_dates.to_period(freq='A')


### 6-2 시계열 데이터 만들기
#### Timstamp 배열

- `pd.date_range(start='시작날짜', end='끝날짜', periods=생성개수, freq='시간간격', tz='시간대')` : 여러 개의 Timestamp가 들어 있는 시계열 데이터를 만듦
    
    
    | 옵션 | 의미 |
    | --- | --- |
    | `start` | 날짜 범위의 시작 |
    | `end` | 날짜 범위의 끝 (`None`이면 지정하지 않음) |
    | `periods` | 생성할 Timestamp의 개수 |
    | `freq` | 시간 간격 (위 `freq` 표 참고, 숫자를 붙이면 배수 간격) |
    | `tz` | 시간대(timezone). `'Asia/Seoul'`은 한국 시간대 |

In [17]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# Timestamp의 배열 만들기 - 월 간격, 월의 시작일 기준
ts_ms = pd.date_range(start='2019-01-01',    # 날짜 범위 시작
                      end=None,              # 날짜 범위 끝
                      periods=6,             # 생성할 Timestamp 개수
                      freq='MS',             # 시간 간격(MS: 월의 시작일)
                      tz='Asia/Seoul')       # 시간대(timezone)
print(ts_ms)


# 월 간격, 월의 마지막 날 기준
ts_me = pd.date_range('2019-01-01', periods=6,
                      freq='M',              # 시간 간격(M: 월의 마지막 날)
                      tz='Asia/Seoul')       # 시간대(timezone)
print(ts_me)
print('\n')

# 분기(3개월) 간격, 월의 마지막 날 기준
ts_3m = pd.date_range('2019-01-01', periods=6,
                      freq='3M',             # 시간 간격(3M: 3개월)
                      tz='Asia/Seoul')       # 시간대(timezone)
print(ts_3m)

DatetimeIndex(['2019-01-01 00:00:00+09:00', '2019-02-01 00:00:00+09:00',
               '2019-03-01 00:00:00+09:00', '2019-04-01 00:00:00+09:00',
               '2019-05-01 00:00:00+09:00', '2019-06-01 00:00:00+09:00'],
              dtype='datetime64[ns, Asia/Seoul]', freq='MS')
DatetimeIndex(['2019-01-31 00:00:00+09:00', '2019-02-28 00:00:00+09:00',
               '2019-03-31 00:00:00+09:00', '2019-04-30 00:00:00+09:00',
               '2019-05-31 00:00:00+09:00', '2019-06-30 00:00:00+09:00'],
              dtype='datetime64[ns, Asia/Seoul]', freq='ME')


DatetimeIndex(['2019-01-31 00:00:00+09:00', '2019-04-30 00:00:00+09:00',
               '2019-07-31 00:00:00+09:00', '2019-10-31 00:00:00+09:00',
               '2020-01-31 00:00:00+09:00', '2020-04-30 00:00:00+09:00'],
              dtype='datetime64[ns, Asia/Seoul]', freq='3ME')


/tmp/ipykernel_1257/3415331391.py:16: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts_me = pd.date_range('2019-01-01', periods=6,
/tmp/ipykernel_1257/3415331391.py:23: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  ts_3m = pd.date_range('2019-01-01', periods=6,


#### Period 배열

- `pd.period_range(start='시작날짜', end='끝날짜', periods=생성개수, freq='기간의길이')` : 여러 개의 기간이 들어 있는 시계열 데이터를 만듦
    
    
    | 옵션 | 의미 |
    | --- | --- |
    | `start` | 날짜 범위의 시작 |
    | `end` | 날짜 범위의 끝 (`None`이면 지정하지 않음) |
    | `periods` | 생성할 Period의 개수 |
    | `freq` | 기간의 길이 (`M`: 월, `H`: 시간, 숫자를 붙이면 배수 길이) |

In [18]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# Period 배열 만들기 - 1개월 길이
pr_m = pd.period_range(start='2019-01-01',    # 날짜 범위 시작
                       end=None,              # 날짜 범위 끝
                       periods=3,             # 생성할 Period 개수
                       freq='M')              # 기간의 길이(M: 월)
print(pr_m)


# Period 배열 만들기 - 1시간 길이
pr_h = pd.period_range(start='2019-01-01',    # 날짜 범위 시작
                       end=None,              # 날짜 범위 끝
                       periods=3,             # 생성할 Period 개수
                       freq='H')              # 기간의 길이(H: 시간)
print(pr_h)
print('\n')

# Period 배열 만들기 - 2시간 길이
pr_2h = pd.period_range(start='2019-01-01',    # 날짜 범위 시작
                        end=None,              # 날짜 범위 끝
                        periods=3,             # 생성할 Period 개수
                        freq='2H')             # 기간의 길이(2H: 2시간)
print(pr_2h)

PeriodIndex(['2019-01', '2019-02', '2019-03'], dtype='period[M]')
PeriodIndex(['2019-01-01 00:00', '2019-01-01 01:00', '2019-01-01 02:00'], dtype='period[h]')


PeriodIndex(['2019-01-01 00:00', '2019-01-01 02:00', '2019-01-01 04:00'], dtype='period[2h]')


/tmp/ipykernel_1257/2571845683.py:15: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pr_h = pd.period_range(start='2019-01-01',    # 날짜 범위 시작
/tmp/ipykernel_1257/2571845683.py:23: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  pr_2h = pd.period_range(start='2019-01-01',    # 날짜 범위 시작


### 6-3 시계열 데이터의 활용

#### 날짜 데이터 분리

- `Series객체.dt.@`
    - dt 속성
    
    | dt  | 의미 |
    | --- | --- |
    | `.year` | 연도 추출해 시리즈로 반환 |
    | `.month` | 월.. |
    | `.day` | 일 |
    | `.to_period(freq=’기간기준’)` | TImestamp →Period로 바꿈 |

In [19]:
# -*- coding: utf-8 -*-

# 라이브러리 불러오기
import pandas as pd

# read_csv() 함수로 파일 읽어와서 df로 변환
df = pd.read_csv('/content/drive/MyDrive/학회/part 5 data/stock-data.csv')

# 문자열인 날짜 데이터를 판다스 Timestamp로 변환
df['new_Date'] = pd.to_datetime(df['Date'])   # df에 새로운 열로 추가
print(df.head())
print('\n')

# dt 속성을 이용하여 new_Date 열의 연-월-일 정보를 년, 월, 일로 구분
df['Year'] = df['new_Date'].dt.year
df['Month'] = df['new_Date'].dt.month
df['Day'] = df['new_Date'].dt.day
print(df.head())

         Date  Close  Start   High    Low  Volume   new_Date
0  2018-07-02  10100  10850  10900  10000  137977 2018-07-02
1  2018-06-29  10700  10550  10900   9990  170253 2018-06-29
2  2018-06-28  10400  10900  10950  10150  155769 2018-06-28
3  2018-06-27  10900  10800  11050  10500  133548 2018-06-27
4  2018-06-26  10800  10900  11000  10700   63039 2018-06-26


         Date  Close  Start   High    Low  Volume   new_Date  Year  Month  Day
0  2018-07-02  10100  10850  10900  10000  137977 2018-07-02  2018      7    2
1  2018-06-29  10700  10550  10900   9990  170253 2018-06-29  2018      6   29
2  2018-06-28  10400  10900  10950  10150  155769 2018-06-28  2018      6   28
3  2018-06-27  10900  10800  11050  10500  133548 2018-06-27  2018      6   27
4  2018-06-26  10800  10900  11000  10700   63039 2018-06-26  2018      6   26


In [20]:
# Timestamp를 Period로 변환하여 연-월-일 표기 변경하기
df['Date_yr'] = df['new_Date'].dt.to_period(freq='A')
df['Date_m'] = df['new_Date'].dt.to_period(freq='M')
print(df.head())

         Date  Close  Start   High    Low  Volume   new_Date  Year  Month  \
0  2018-07-02  10100  10850  10900  10000  137977 2018-07-02  2018      7   
1  2018-06-29  10700  10550  10900   9990  170253 2018-06-29  2018      6   
2  2018-06-28  10400  10900  10950  10150  155769 2018-06-28  2018      6   
3  2018-06-27  10900  10800  11050  10500  133548 2018-06-27  2018      6   
4  2018-06-26  10800  10900  11000  10700   63039 2018-06-26  2018      6   

   Day Date_yr   Date_m  
0    2    2018  2018-07  
1   29    2018  2018-06  
2   28    2018  2018-06  
3   27    2018  2018-06  
4   26    2018  2018-06  


/tmp/ipykernel_1257/2517450149.py:2: FutureWarning: 'A' is deprecated and will be removed in a future version, please use 'Y' instead.
  df['Date_yr'] = df['new_Date'].dt.to_period(freq='A')


In [21]:
# 원하는 열을 새로운 행 인덱스로 지정
df.set_index('Date_m', inplace=True)
print(df.head())

               Date  Close  Start   High    Low  Volume   new_Date  Year  \
Date_m                                                                     
2018-07  2018-07-02  10100  10850  10900  10000  137977 2018-07-02  2018   
2018-06  2018-06-29  10700  10550  10900   9990  170253 2018-06-29  2018   
2018-06  2018-06-28  10400  10900  10950  10150  155769 2018-06-28  2018   
2018-06  2018-06-27  10900  10800  11050  10500  133548 2018-06-27  2018   
2018-06  2018-06-26  10800  10900  11000  10700   63039 2018-06-26  2018   

         Month  Day Date_yr  
Date_m                       
2018-07      7    2    2018  
2018-06      6   29    2018  
2018-06      6   28    2018  
2018-06      6   27    2018  
2018-06      6   26    2018  


#### 날짜 인덱스 활용

In [22]:
# 문자열인 날짜 데이터를 판다스 Timestamp로 변환
df['new_Date'] = pd.to_datetime(df['Date'])   # 새로운 열로 추가
df.set_index('new_Date', inplace=True)   # 행 인덱스로 지정

print(df.head())
print('\n')
print(df.index)

                  Date  Close  Start   High    Low  Volume  Year  Month  Day  \
new_Date                                                                       
2018-07-02  2018-07-02  10100  10850  10900  10000  137977  2018      7    2   
2018-06-29  2018-06-29  10700  10550  10900   9990  170253  2018      6   29   
2018-06-28  2018-06-28  10400  10900  10950  10150  155769  2018      6   28   
2018-06-27  2018-06-27  10900  10800  11050  10500  133548  2018      6   27   
2018-06-26  2018-06-26  10800  10900  11000  10700   63039  2018      6   26   

           Date_yr  
new_Date            
2018-07-02    2018  
2018-06-29    2018  
2018-06-28    2018  
2018-06-27    2018  
2018-06-26    2018  


DatetimeIndex(['2018-07-02', '2018-06-29', '2018-06-28', '2018-06-27',
               '2018-06-26', '2018-06-25', '2018-06-22', '2018-06-21',
               '2018-06-20', '2018-06-19', '2018-06-18', '2018-06-15',
               '2018-06-14', '2018-06-12', '2018-06-11', '2018-06-08',
      

In [35]:
# -*- coding: utf-8 -*-
import pandas as pd

# 안전하게 데이터를 다시 로드하고 시계열 인덱스로 설정합니다.
df = pd.read_csv('/content/drive/MyDrive/학회/part 5 data/stock-data.csv')
df['new_Date'] = pd.to_datetime(df['Date'])
df.set_index('new_Date', inplace=True)

# 슬라이싱(Slicing)을 위해 인덱스를 오름차순으로 정렬합니다.
df.sort_index(inplace=True)

# 날짜 인덱스를 이용하여 데이터 선택하기
df_y = df.loc['2018']
print(df_y.head())
print('\n')
df_ym = df.loc['2018-07']   # loc 인덱서 활용
print(df_ym)
print('\n')
df_ym_cols = df.loc['2018-07', 'Start':'High']   # 열 범위 슬라이싱
print(df_ym_cols)
print('\n')
df_ymd = df.loc['2018-07-02':'2018-07-02']
print(df_ymd)
print('\n')
# 인덱스가 오름차순이므로 과거 날짜에서 최신 날짜 순으로 범위를 지정해야 정상 작동합니다.
df_ymd_range = df['2018-06-20':'2018-06-25']   # 날짜 범위 순서 조정
print(df_ymd_range)

                  Date  Close  Start   High    Low  Volume
new_Date                                                  
2018-06-01  2018-06-01  11900  11800  12100  11750   32062
2018-06-04  2018-06-04  11900  11900  12200  11700   25171
2018-06-05  2018-06-05  12150  11800  12250  11800   42485
2018-06-07  2018-06-07  11950  12200  12300  11900   49088
2018-06-08  2018-06-08  11950  11950  12200  11800   59258


                  Date  Close  Start   High    Low  Volume
new_Date                                                  
2018-07-02  2018-07-02  10100  10850  10900  10000  137977


            Start   High
new_Date                
2018-07-02  10850  10900


                  Date  Close  Start   High    Low  Volume
new_Date                                                  
2018-07-02  2018-07-02  10100  10850  10900  10000  137977


                  Date  Close  Start   High    Low  Volume
new_Date                                                  
2018-06-20  2018-06-20  11550  1

In [37]:
import pandas as pd

# read_csv() 함수로 파일 읽어와서 df로 변환
df = pd.read_csv('/content/drive/MyDrive/학회/part 5 data/stock-data.csv')

# 문자열인 날짜 데이터를 판다스 Timestamp로 변환
df['new_Date'] = pd.to_datetime(df['Date'])   # 새로운 열로 추가
df.set_index('new_Date', inplace=True)   # 행 인덱스로 지정

# 시간 간격 계산. 최근 180일~189일 사이의 값들만 선택하기
today = pd.to_datetime('2018-12-25')   # 기준일 생성
df['time_delta'] = today - df.index   # 날짜 차이 계산
df.set_index('time_delta', inplace=True)   # 행 인덱스로 지정
df_180 = df['180 days':'189 days']
print(df_180)

                  Date  Close  Start   High    Low  Volume
time_delta                                                
180 days    2018-06-28  10400  10900  10950  10150  155769
181 days    2018-06-27  10900  10800  11050  10500  133548
182 days    2018-06-26  10800  10900  11000  10700   63039
183 days    2018-06-25  11150  11400  11450  11000   55519
186 days    2018-06-22  11300  11250  11450  10750  134805
187 days    2018-06-21  11200  11350  11750  11200  133002
188 days    2018-06-20  11550  11200  11600  10900  308596
189 days    2018-06-19  11300  11850  11950  11300  180656
